# Apple 재무지표 수집

- project: U.S. Financial Research Project
- company: Apple Inc.
- ticker: `AAPL`
- CIK: `0000320193`
- source: SEC EDGAR companyfacts
- fiscal year range: 2021~2025
- output notebook: `notebooks/08_sec_apple_metric.ipynb`
- output CSV: `data/apple_financials.csv`

## 작업 목표

1. `us-gaap`에서 Apple 전체 매출과 순이익 concept을 확인한다.
2. 10-K·FY·연차 기간 조건으로 annual facts를 선택하고 비교기간 반복 record를 정리한다.
3. 매출과 순이익을 회계연도 기준으로 병합해 CSV로 저장하고 다시 읽어 검수한다.

매출과 순이익은 companyfacts의 구조화된 XBRL facts다. 값의 원인을 설명하려면 실제 10-K 원문과 재무제표 주석을 추가로 확인해야 한다.

In [1]:
from getpass import getpass
from pathlib import Path
import time

import pandas as pd
import requests
from IPython.display import display

current_path = Path.cwd().resolve()
candidate_roots = [current_path, *current_path.parents]

project_root = next(
    (
        path
        for path in candidate_roots
        if (path / "notebooks").is_dir()
        and (path / "data").is_dir()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "notebooks의 data 폴더가 있는 프로젝트 루트를 찾지 못했습니다."
    )

notebook_output_path = (
    project_root
    / "notebooks"
    / "08_sec_apple_metric.ipynb"
)

csv_output_path = (
    project_root
    / "data"
    / "apple_financials.csv"
)

print("project_root:", project_root)
print("notebook_output_path:", notebook_output_path)
print("csv_output_path:", csv_output_path)

project_root: /Users/im-youngchan/Desktop/US Financial
notebook_output_path: /Users/im-youngchan/Desktop/US Financial/notebooks/08_sec_apple_metric.ipynb
csv_output_path: /Users/im-youngchan/Desktop/US Financial/data/apple_financials.csv


In [2]:
APPLE_TICKER = "AAPL"
apple_cik_int = 320193
apple_cik = f"{apple_cik_int:010d}"

SEC_COMPANYFACTS_URL_TEMPLATE = (
    "https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
)
REQUEST_TIMEOUT_SECONDS = 30
REQUEST_INTERVAL_SECONDS = 0.2

FISCAL_YEAR_START = 2021
FISCAL_YEAR_END = 2025

REVENUE_CONCEPT_ID = (
    "RevenueFromContractWithCustomerExcludingAssessedTax"
)
NET_INCOME_CONCEPT_ID = "NetIncomeLoss"

print("ticker:", APPLE_TICKER)
print("apple_cik_int:", apple_cik_int)
print("apple_cik:", apple_cik)
print(
    "fiscal_year_range:",
    FISCAL_YEAR_START,
    "to",
    FISCAL_YEAR_END,
)
print("request_timeout_seconds:", REQUEST_TIMEOUT_SECONDS)
print("request_interval_seconds:", REQUEST_INTERVAL_SECONDS)

ticker: AAPL
apple_cik_int: 320193
apple_cik: 0000320193
fiscal_year_range: 2021 to 2025
request_timeout_seconds: 30
request_interval_seconds: 0.2


In [3]:
sec_contract_email = getpass(
    "SEC User-Agent에 넣을 연락처 이메일을 입력하세요: "
).strip()

if not sec_contract_email or "@" not in sec_contract_email:
    raise ValueError("연락가능한 이메일 형식을 입력하세요.")

sec_user_agent = (
    f"U.S. Financial Research Project {sec_contract_email}"
)
sec_headers = {
    "User-Agent": sec_user_agent,
    "Accept-Encoding": "gzip, deflate",
}

print("sec_user_agent_configured:", True)
print("sec_header_names:", list(sec_headers.keys()))

sec_user_agent_configured: True
sec_header_names: ['User-Agent', 'Accept-Encoding']


In [4]:
def get_sec_json(
        url,
        headers,
        pause_seconds=REQUEST_INTERVAL_SECONDS,
):
    if pause_seconds < 0:
        raise ValueError("pause_seconds는 0 이상이어야 합니다.")
    
    time.sleep(pause_seconds)

    response = requests.get(
        url,
        headers=headers,
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    response.raise_for_status()

    payload = response.json()

    if not isinstance(payload, dict):
        raise TypeError("SEC JSON 최상위 구조가 딕셔너리가 아닙니다.")
    
    return payload

In [5]:
companyfacts_url = SEC_COMPANYFACTS_URL_TEMPLATE.format(
    cik=apple_cik
)

apple_companyfacts_json = get_sec_json(
    url=companyfacts_url,
    headers=sec_headers,
)

required_top_level_keys = {
    "cik",
    "entityName",
    "facts",
}
missing_top_level_keys = (
    required_top_level_keys
    - set(apple_companyfacts_json.keys())
)

if missing_top_level_keys:
    raise ValueError(
        "companyfacts 최상위 key가 부족합니다: "
        f"{sorted(missing_top_level_keys)}"
    )

if int(apple_companyfacts_json["cik"]) != apple_cik_int:
    raise ValueError("companyfacts CIK가 Apple CIK와 다릅니다.")

print("companyfacts_cik:", apple_companyfacts_json["cik"])
print(
    "compnayfacts_entity_name:",
    apple_companyfacts_json["entityName"],
)
print(
    "companyfacts_top_level_keys:",
    sorted(apple_companyfacts_json.keys())
)

companyfacts_cik: 320193
compnayfacts_entity_name: Apple Inc.
companyfacts_top_level_keys: ['cik', 'entityName', 'facts']


In [6]:
facts_by_taxonomy = apple_companyfacts_json["facts"]

if "us-gaap" not in facts_by_taxonomy:
    raise KeyError("companyfacts에서 us-gaap taxonomy를 찾지 못했습니다.")

us_gaap_facts = facts_by_taxonomy["us-gaap"]

print("taxonomy_names:", sorted(facts_by_taxonomy.keys()))
print("us_gaap_concept_count:", len(us_gaap_facts))

taxonomy_names: ['dei', 'us-gaap']
us_gaap_concept_count: 503


In [8]:
def find_concept_candidates(
        facts,
        keywords,
):
    normalized_keywords = tuple(
        keyword.lower()
        for keyword in keywords
    )
    candidate_rows = []

    for concept_id, concept_data in facts.items():
        concept_text = " ".join(
            [
                concept_id,
                str(concept_data.get("label", ""))
            ]
        ).lower()

        if not any(
            keyword in concept_text
            for keyword in normalized_keywords
        ):
            continue

        candidate_rows.append(
            {
                "concept_id": concept_id,
                "label": concept_data.get("label"),
                "unit_names": ", ".join(
                    sorted(
                        concept_data
                        .get("units", {})
                        .keys()
                    )
                ),
                "description": concept_data.get("description"),
            }
        )

    candidate_columns = [
        "concept_id",
        "label",
        "unit_names",
        "description"
    ]
    candidate_df = pd.DataFrame(
        candidate_rows,
        columns=candidate_columns,
    )

    return (
        candidate_df
        .sort_values("concept_id")
        .reset_index(drop=True)
    )

In [9]:
revenue_candidate_df = find_concept_candidates(
    facts=us_gaap_facts,
    keywords=("Revenue", "Sales"),
)

print(
    "revenue_candidate_count:",
    len(revenue_candidate_df), 
)
display(
    revenue_candidate_df[
        ["concept_id", "label", "unit_names"]
    ].head(30)
)

revenue_candidate_count: 39


,concept_id,label,unit_names
0,AccumulatedOtherComprehensiveIncomeLossAvailab...,"AOCI, Debt Securities, Available-for-sale, Adj...",USD
1,AvailableForSaleSecurities,Available-for-sale Securities,USD
2,AvailableForSaleSecuritiesAccumulatedGrossUnre...,"Available-for-sale Securities, Accumulated Gro...",USD
3,AvailableForSaleSecuritiesAccumulatedGrossUnre...,"Available-for-sale Securities, Accumulated Gro...",USD
4,AvailableForSaleSecuritiesAmortizedCost,"Available-for-sale Securities, Amortized Cost ...",USD
5,AvailableForSaleSecuritiesContinuousUnrealized...,"Available-for-sale Securities, Continuous Unre...",USD
6,AvailableForSaleSecuritiesContinuousUnrealized...,"Available-for-sale Securities, Continuous Unre...",USD
7,AvailableForSaleSecuritiesContinuousUnrealized...,"Available-for-sale Securities, Continuous Unre...",USD
8,AvailableForSaleSecuritiesContinuousUnrealized...,"Available-for-sale Securities, Continuous Unre...",USD
9,AvailableForSaleSecuritiesContinuousUnrealized...,"Available-for-sale Securities, Continuous Unre...",USD


In [11]:
net_income_candidate_df = find_concept_candidates(
    facts=us_gaap_facts,
    keywords=("Revenue", "Sales"),
)

print(
    "net_income_candidate_df:",
    len(revenue_candidate_df), 
)
display(
    net_income_candidate_df[
        ["concept_id", "label", "unit_names"]
    ].head(30)
)

net_income_candidate_df: 39


,concept_id,label,unit_names
0,AccumulatedOtherComprehensiveIncomeLossAvailab...,"AOCI, Debt Securities, Available-for-sale, Adj...",USD
1,AvailableForSaleSecurities,Available-for-sale Securities,USD
2,AvailableForSaleSecuritiesAccumulatedGrossUnre...,"Available-for-sale Securities, Accumulated Gro...",USD
3,AvailableForSaleSecuritiesAccumulatedGrossUnre...,"Available-for-sale Securities, Accumulated Gro...",USD
4,AvailableForSaleSecuritiesAmortizedCost,"Available-for-sale Securities, Amortized Cost ...",USD
5,AvailableForSaleSecuritiesContinuousUnrealized...,"Available-for-sale Securities, Continuous Unre...",USD
6,AvailableForSaleSecuritiesContinuousUnrealized...,"Available-for-sale Securities, Continuous Unre...",USD
7,AvailableForSaleSecuritiesContinuousUnrealized...,"Available-for-sale Securities, Continuous Unre...",USD
8,AvailableForSaleSecuritiesContinuousUnrealized...,"Available-for-sale Securities, Continuous Unre...",USD
9,AvailableForSaleSecuritiesContinuousUnrealized...,"Available-for-sale Securities, Continuous Unre...",USD


In [12]:
metric_concepts = {
    "revenue": REVENUE_CONCEPT_ID,
    "net_income": NET_INCOME_CONCEPT_ID,
}

concept_summary_rows = []

for metric_name, concept_id in metric_concepts.items():
    if concept_id not in us_gaap_facts:
        raise KeyError(
            f"us-gaap에서 {concept_id}를 찾지 못했ㅅ브니다."
        )

    concept_data = us_gaap_facts[concept_id]
    unit_names = sorted(
        concept_data.get("units", {}).keys()
    )

    if "USD" not in unit_names:
        raise KeyError(
            f"{concept_id}에서 USD unit를 찾지 못했습니다."
        )

    concept_summary_rows.append(
        {
            "metric_name": metric_name,
            "concept_id": concept_id,
            "label": concept_data.get("label"),
            "unit_names": ", ".join(unit_names),
            "description": concept_data.get("description"),
        }
    )

concept_summary_df = pd.DataFrame(concept_summary_rows)

display(concept_summary_df)

,metric_name,concept_id,label,unit_names,description
0,revenue,RevenueFromContractWithCustomerExcludingAssess...,"Revenue from Contract with Customer, Excluding...",USD,"Amount, excluding tax collected from customer,..."
1,net_income,NetIncomeLoss,Net Income (Loss) Attributable to Parent,USD,"The portion of profit or loss for the period, ..."


## concept 선택

- Apple 연결 전체 매출은 `us-gaap`의 `RevenueFromContractWithCustomerExcludingAssessedTax` concept으로 확인한다.
- 연결 순이익은 `us-gaap`의 `NetIncomeLoss` concept으로 확인한다.
- 두 지표 모두 `units -> USD` records를 사용한다.

In [15]:
def concept_usd_facts_to_df(
        companyfacts_json,
        concept_id,
):
    try:
        concept_data = (
            companyfacts_json["facts"]
            ["us-gaap"]
            [concept_id]
        )
        usd_records = concept_data["units"]["USD"]
    except KeyError as error:
        raise KeyError(
            f"{concept_id}의 USD facts 경로를 찾지 못했습니다."
        ) from error

    facts_df = pd.DataFrame(usd_records)

    required_columns = [
        "start",
        "end",
        "val",
        "accn",
        "fy",
        "fp",
        "form",
        "filed",
    ]
    missing_columns = [
        column
        for column in required_columns
        if column not in facts_df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{concept_id} facts에 필요한 column이 없습니다: "
            f"{missing_columns}"
        )

    facts_df["start"] = pd.to_datetime(
        facts_df["start"],
        errors="coerce",
    )
    facts_df["end"] = pd.to_datetime(
        facts_df["end"],
        errors="coerce",
    )
    facts_df["filed"] = pd.to_datetime(
        facts_df["filed"],
        errors="coerce",
    )
    facts_df["val"] = pd.to_numeric(
        facts_df["val"],
        errors="coerce",
    )

    facts_df["duration_days"] = (
        facts_df["end"]
        - facts_df["start"]
    ).dt.days + 1
    facts_df["concept_id"] = concept_id
    facts_df["unit"] = "USD"
    return facts_df

In [16]:
revenue_facts_df = concept_usd_facts_to_df(
    companyfacts_json=apple_companyfacts_json,
    concept_id=REVENUE_CONCEPT_ID,
)
net_income_facts_df = concept_usd_facts_to_df(
    companyfacts_json=apple_companyfacts_json,
    concept_id=NET_INCOME_CONCEPT_ID,
)

print("revenue_raw_fact_count:", len(revenue_facts_df))
print(
    "net_income_raw_fact_count:",
    len(net_income_facts_df),
)
print(
    "revenue_missing_value_count:",
    revenue_facts_df["val"].isna().sum(),
)
print(
    "net_income_missing_value_count:",
    net_income_facts_df["val"].isna().sum(),
)

revenue_raw_fact_count: 117
net_income_raw_fact_count: 338
revenue_missing_value_count: 0
net_income_missing_value_count: 0


In [17]:
annual_candidate_columns = [
    "start",
    "end",
    "val",
    "duration_days",
    "form",
    "fy",
    "fp",
    "filed",
    "accn",
    "frame",
]

available_revenue_columns = [
    column
    for column in annual_candidate_columns
    if column in revenue_facts_df.columns
]

revenue_10k_fy_candidate_df = (
    revenue_facts_df.loc[
        revenue_facts_df["form"].eq("10-K")
        & revenue_facts_df["fp"].eq("FY")
    ]
    .sort_values(
        ["end", "filed"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

print(
    "revenue_10k_fy_candidate_count:",
    len(revenue_10k_fy_candidate_df),
)
display(
    revenue_10k_fy_candidate_df[
        available_revenue_columns
    ].head(20)
)

revenue_10k_fy_candidate_count: 37


,start,end,val,duration_days,form,fy,fp,filed,accn,frame
0,2024-09-29,2025-09-27,416161000000,364,10-K,2025,FY,2025-10-31,0000320193-25-000079,CY2025
1,2023-10-01,2024-09-28,391035000000,364,10-K,2025,FY,2025-10-31,0000320193-25-000079,CY2024
2,2023-10-01,2024-09-28,391035000000,364,10-K,2024,FY,2024-11-01,0000320193-24-000123,NaN
3,2022-09-25,2023-09-30,383285000000,371,10-K,2025,FY,2025-10-31,0000320193-25-000079,CY2023
4,2022-09-25,2023-09-30,383285000000,371,10-K,2024,FY,2024-11-01,0000320193-24-000123,NaN
5,2022-09-25,2023-09-30,383285000000,371,10-K,2023,FY,2023-11-03,0000320193-23-000106,NaN
6,2021-09-26,2022-09-24,394328000000,364,10-K,2024,FY,2024-11-01,0000320193-24-000123,CY2022
7,2021-09-26,2022-09-24,394328000000,364,10-K,2023,FY,2023-11-03,0000320193-23-000106,NaN
8,2021-09-26,2022-09-24,394328000000,364,10-K,2022,FY,2022-10-28,0000320193-22-000108,NaN
9,2020-09-27,2021-09-25,365817000000,364,10-K,2023,FY,2023-11-03,0000320193-23-000106,CY2021


In [19]:
def extract_annual_usd_facts(
        companyfacts_json,
        concept_id,
        metric_name,
        fiscal_year_start,
        fiscal_year_end,
):
    facts_df = concept_usd_facts_to_df(
        companyfacts_json=companyfacts_json,
        concept_id=concept_id,
    )

    annual_df = facts_df.loc[
        facts_df["form"].eq("10-K")
        & facts_df["fp"].eq("FY")
        & facts_df["duration_days"].between(330, 400)
    ].copy()

    annual_df = annual_df.dropna(
        subset=["start", "end", "filed", "val"]
    )
    annual_df["fiscal_year"] = (
        annual_df["end"].dt.year
    )
    annual_df = annual_df.loc[
        annual_df["fiscal_year"].between(
            fiscal_year_start,
            fiscal_year_end,
        )
    ].copy()

    if annual_df.empty:
        raise ValueError(
            f"{concept_id}에서 지정 범위의 annual facts를 찾지 못했습니다."
        )

    annual_df = (
        annual_df
        .sort_values(
            ["end", "filed", "accn"],
            ascending=[True, True, True],
        )
        .drop_duplicates(
            subset=["fiscal_year"],
            keep="last",
        )
        .sort_values("fiscal_year")
        .reset_index(drop=True)
    )

    annual_df = annual_df.rename(
        columns={
            "start": f"{metric_name}_period_start",
            "end": f"{metric_name}_period_end",
            "val": f"{metric_name}_usd",
            "filed": f"{metric_name}_filed",
            "accn": f"{metric_name}_accn",
            "duration_days": f"{metric_name}_duration_days",
            "concept_id": f"{metric_name}_concept",
        }
    )

    output_columns = [
        "fiscal_year",
        f"{metric_name}_period_start",
        f"{metric_name}_period_end",
        f"{metric_name}_usd",
        f"{metric_name}_filed",
        f"{metric_name}_accn",
        f"{metric_name}_duration_days",
        f"{metric_name}_concept",
    ]

    return annual_df[output_columns]

In [20]:
revenue_annual_df = extract_annual_usd_facts(
    companyfacts_json=apple_companyfacts_json,
    concept_id=REVENUE_CONCEPT_ID,
    metric_name="revenue",
    fiscal_year_start=FISCAL_YEAR_START,
    fiscal_year_end=FISCAL_YEAR_END,
)
net_income_annual_df = extract_annual_usd_facts(
    companyfacts_json=apple_companyfacts_json,
    concept_id=NET_INCOME_CONCEPT_ID,
    metric_name="net_income",
    fiscal_year_start=FISCAL_YEAR_START,
    fiscal_year_end=FISCAL_YEAR_END,
)

print("revenue_annual_row_count:", len(revenue_annual_df))
print(
    "net_income_annual_row_count:",
    len(net_income_annual_df),
)

display(revenue_annual_df)
display(net_income_annual_df)

revenue_annual_row_count: 5
net_income_annual_row_count: 5


,fiscal_year,revenue_period_start,revenue_period_end,revenue_usd,revenue_filed,revenue_accn,revenue_duration_days,revenue_concept
0,2021,2020-09-27,2021-09-25,365817000000,2023-11-03,0000320193-23-000106,364,RevenueFromContractWithCustomerExcludingAssess...
1,2022,2021-09-26,2022-09-24,394328000000,2024-11-01,0000320193-24-000123,364,RevenueFromContractWithCustomerExcludingAssess...
2,2023,2022-09-25,2023-09-30,383285000000,2025-10-31,0000320193-25-000079,371,RevenueFromContractWithCustomerExcludingAssess...
3,2024,2023-10-01,2024-09-28,391035000000,2025-10-31,0000320193-25-000079,364,RevenueFromContractWithCustomerExcludingAssess...
4,2025,2024-09-29,2025-09-27,416161000000,2025-10-31,0000320193-25-000079,364,RevenueFromContractWithCustomerExcludingAssess...


,fiscal_year,net_income_period_start,net_income_period_end,net_income_usd,net_income_filed,net_income_accn,net_income_duration_days,net_income_concept
0,2021,2020-09-27,2021-09-25,94680000000,2023-11-03,0000320193-23-000106,364,NetIncomeLoss
1,2022,2021-09-26,2022-09-24,99803000000,2024-11-01,0000320193-24-000123,364,NetIncomeLoss
2,2023,2022-09-25,2023-09-30,96995000000,2025-10-31,0000320193-25-000079,371,NetIncomeLoss
3,2024,2023-10-01,2024-09-28,93736000000,2025-10-31,0000320193-25-000079,364,NetIncomeLoss
4,2025,2024-09-29,2025-09-27,112010000000,2025-10-31,0000320193-25-000079,364,NetIncomeLoss


In [21]:
merged_financials_df = revenue_annual_df.merge(
    net_income_annual_df,
    on="fiscal_year",
    how="outer",
    validate="one_to_one",
    indicator=True,
)

if not merged_financials_df["_merge"].eq("both").all():
    display(
        merged_financials_df.loc[
            merged_financials_df["_merge"].ne("both")
        ]
    )
    raise ValueError(
        "revenue와 net income의 fiscal year 범위가 일치하지 않습니다."
    )

merged_financials_df = merged_financials_df.drop(
    columns="_merge"
)

print("merged_row_count:", len(merged_financials_df))
display(merged_financials_df)

merged_row_count: 5


,fiscal_year,revenue_period_start,revenue_period_end,revenue_usd,revenue_filed,revenue_accn,revenue_duration_days,revenue_concept,net_income_period_start,net_income_period_end,net_income_usd,net_income_filed,net_income_accn,net_income_duration_days,net_income_concept
0,2021,2020-09-27,2021-09-25,365817000000,2023-11-03,0000320193-23-000106,364,RevenueFromContractWithCustomerExcludingAssess...,2020-09-27,2021-09-25,94680000000,2023-11-03,0000320193-23-000106,364,NetIncomeLoss
1,2022,2021-09-26,2022-09-24,394328000000,2024-11-01,0000320193-24-000123,364,RevenueFromContractWithCustomerExcludingAssess...,2021-09-26,2022-09-24,99803000000,2024-11-01,0000320193-24-000123,364,NetIncomeLoss
2,2023,2022-09-25,2023-09-30,383285000000,2025-10-31,0000320193-25-000079,371,RevenueFromContractWithCustomerExcludingAssess...,2022-09-25,2023-09-30,96995000000,2025-10-31,0000320193-25-000079,371,NetIncomeLoss
3,2024,2023-10-01,2024-09-28,391035000000,2025-10-31,0000320193-25-000079,364,RevenueFromContractWithCustomerExcludingAssess...,2023-10-01,2024-09-28,93736000000,2025-10-31,0000320193-25-000079,364,NetIncomeLoss
4,2025,2024-09-29,2025-09-27,416161000000,2025-10-31,0000320193-25-000079,364,RevenueFromContractWithCustomerExcludingAssess...,2024-09-29,2025-09-27,112010000000,2025-10-31,0000320193-25-000079,364,NetIncomeLoss


In [22]:
period_match_mask = (
    merged_financials_df["revenue_period_start"].eq(
        merged_financials_df["net_income_period_start"]
    )
    & merged_financials_df["revenue_period_end"].eq(
        merged_financials_df["revenue_period_end"]
    )
)

if not period_match_mask.all():
    period_mismatch_df = merged_financials_df.loc[
        ~period_match_mask,
        [
            "fiscal_year",
            "revenue_period_start",
            "revenue_period_end",
            "net_income_period_start",
            "net_income_period_end",
        ],
    ]
    display(period_mismatch_df)
    raise ValueError(
        "revenue와 net income의 보고 기간이 일치하지 않습니다."
    )

print("all_metric_periods_match:", True)

all_metric_periods_match: True


In [34]:
apple_financials_df = merged_financials_df.copy()

apple_financials_df["company_name"] = (
    apple_companyfacts_json["entityName"]
)
apple_financials_df["ticker"] = APPLE_TICKER
apple_financials_df["cik"] = apple_cik
apple_financials_df["period_start"] = (
    apple_financials_df["revenue_period_start"]
)
apple_financials_df["period_end"] = (
    apple_financials_df["revenue_period_end"]
)
apple_financials_df["form"] = "10-K"
apple_financials_df["unit"] = "USD"
apple_financials_df["source"] = (
    "SEC EDGAR companyfacts"
)

apple_financials_df = apple_financials_df.drop(
    columns=[
        "revenue_period_start",
        "revenue_period_end",
        "net_income_period_start",
        "net_income_period_end",
    ]
)

final_columns = [
    "company_name",
    "ticker",
    "cik",
    "fiscal_year",
    "period_start",
    "period_end",
    "revenue_usd",
    "net_income_usd",
    "form",
    "unit",
    "revenue_concept",
    "net_income_concept",
    "revenue_duration_days",
    "net_income_duration_days",
    "revenue_filed",
    "net_income_filed",
    "revenue_accn",
    "net_income_accn",
    "source",
]

apple_financials_df = (
    apple_financials_df[final_columns]
    .sort_values("fiscal_year")
    .reset_index(drop=True)
)

print("final_shape:", apple_financials_df.shape)
display(apple_financials_df)

final_shape: (5, 19)


,company_name,ticker,cik,fiscal_year,period_start,period_end,revenue_usd,net_income_usd,form,unit,revenue_concept,net_income_concept,revenue_duration_days,net_income_duration_days,revenue_filed,net_income_filed,revenue_accn,net_income_accn,source
0,Apple Inc.,AAPL,0000320193,2021,2020-09-27,2021-09-25,365817000000,94680000000,10-K,USD,RevenueFromContractWithCustomerExcludingAssess...,NetIncomeLoss,364,364,2023-11-03,2023-11-03,0000320193-23-000106,0000320193-23-000106,SEC EDGAR companyfacts
1,Apple Inc.,AAPL,0000320193,2022,2021-09-26,2022-09-24,394328000000,99803000000,10-K,USD,RevenueFromContractWithCustomerExcludingAssess...,NetIncomeLoss,364,364,2024-11-01,2024-11-01,0000320193-24-000123,0000320193-24-000123,SEC EDGAR companyfacts
2,Apple Inc.,AAPL,0000320193,2023,2022-09-25,2023-09-30,383285000000,96995000000,10-K,USD,RevenueFromContractWithCustomerExcludingAssess...,NetIncomeLoss,371,371,2025-10-31,2025-10-31,0000320193-25-000079,0000320193-25-000079,SEC EDGAR companyfacts
3,Apple Inc.,AAPL,0000320193,2024,2023-10-01,2024-09-28,391035000000,93736000000,10-K,USD,RevenueFromContractWithCustomerExcludingAssess...,NetIncomeLoss,364,364,2025-10-31,2025-10-31,0000320193-25-000079,0000320193-25-000079,SEC EDGAR companyfacts
4,Apple Inc.,AAPL,0000320193,2025,2024-09-29,2025-09-27,416161000000,112010000000,10-K,USD,RevenueFromContractWithCustomerExcludingAssess...,NetIncomeLoss,364,364,2025-10-31,2025-10-31,0000320193-25-000079,0000320193-25-000079,SEC EDGAR companyfacts


In [35]:
expected_fiscal_year = set(
    range(
        FISCAL_YEAR_START,
        FISCAL_YEAR_END + 1 ,
    )
)
actual_fiscal_years = set(
    apple_financials_df["fiscal_year"].tolist()
)
missing_fiscal_years = sorted(
    expected_fiscal_year - actual_fiscal_years
)
unexpected_fiscal_years = sorted(
    actual_fiscal_years - expected_fiscal_year
)

print("missng_fiscal_years:", missing_fiscal_years)
print("unexpected_fiscal_years:", unexpected_fiscal_years)
print(
    "duplicate_fiscal_year_count:",
    apple_financials_df["fiscal_year"]
    .duplicated()
    .sum()
)
print(
    "missing_value_count:",
    apple_financials_df[
        ["revenue_usd", "net_income_usd"]
    ]
    .isna()
    .sum()
    .to_dict(),
)

if missing_fiscal_years or unexpected_fiscal_years:
    raise ValueError("회계연도 범위가 예상과 다릅니다.")

if apple_financials_df["fiscal_year"].duplicated().any():
    raise ValueError("fiscal_year 중복이 있습니다.")

if apple_financials_df[
    ["revenue_usd", "net_income_usd"]
].isna().any().any():
    raise ValueError("매출 또는 순이익에 결측치가 있습니다.")


missng_fiscal_years: []
unexpected_fiscal_years: []
duplicate_fiscal_year_count: 0
missing_value_count: {'revenue_usd': 0, 'net_income_usd': 0}


In [36]:
display_financials_df = (
    apple_financials_df[
        [
            "fiscal_year",
            "period_start",
            "period_end",
            "revenue_usd",
            "net_income_usd",
        ]
    ]
    .copy()
)

display_financials_df["revenue_usd_b"] = (
    display_financials_df["revenue_usd"]
    / 1_000_000_000
)

display_financials_df["net_income_usd_b"] = (
    display_financials_df["net_income_usd"]
    / 1_000_000_000
)

print("display_unit: USD billion")
display(
    display_financials_df[
        [
            "fiscal_year",
            "period_start",
            "period_end",
            "revenue_usd_b",
            "net_income_usd_b",
        ]
    ].round(3)
)

display_unit: USD billion


,fiscal_year,period_start,period_end,revenue_usd_b,net_income_usd_b
0,2021,2020-09-27,2021-09-25,365.817,94.680
1,2022,2021-09-26,2022-09-24,394.328,99.803
2,2023,2022-09-25,2023-09-30,383.285,96.995
3,2024,2023-10-01,2024-09-28,391.035,93.736
4,2025,2024-09-29,2025-09-27,416.161,112.010


In [37]:
reference_financials_df = pd.DataFrame(
    {
        "fiscal_year": [2021, 2022, 2023, 2024, 2025],
        "reference_revenue_usd": [
            365_817_000_000,
            394_328_000_000,
            383_285_000_000,
            391_035_000_000,
            416_161_000_000,
        ],
        "reference_net_income_usd": [
            94_680_000_000,
            99_803_000_000,
            96_995_000_000,
            93_736_000_000,
            112_010_000_000,
        ],
    }
)

reference_check_df = apple_financials_df.merge(
    reference_financials_df,
    on="fiscal_year",
    how="left",
    validate="one_to_one",
)

reference_check_df["revenue_matches_reference"] = (
    reference_check_df["revenue_usd"].eq(
        reference_check_df["reference_revenue_usd"]
    )
)
reference_check_df["net_income_matches_reference"] = (
    reference_check_df["net_income_usd"].eq(
        reference_check_df["reference_net_income_usd"]
    )
)

display(
    reference_check_df[
        [
            "fiscal_year",
            "revenue_matches_reference",
            "net_income_matches_reference",
            "revenue_accn",
            "net_income_accn",
        ]
    ]
)

,fiscal_year,revenue_matches_reference,net_income_matches_reference,revenue_accn,net_income_accn
0,2021,True,True,0000320193-23-000106,0000320193-23-000106
1,2022,True,True,0000320193-24-000123,0000320193-24-000123
2,2023,True,True,0000320193-25-000079,0000320193-25-000079
3,2024,True,True,0000320193-25-000079,0000320193-25-000079
4,2025,True,True,0000320193-25-000079,0000320193-25-000079


In [38]:
if not csv_output_path.parent.is_dir():
    raise FileNotFoundError(
        "data 폴더가 없습니다. 저장 위치를 확인하세요."
    )

apple_financials_df.to_csv(
    csv_output_path,
    index=False,
    date_format="%Y-%m-%d",
)

print("saved_csv_path:", csv_output_path)
print("saved_csv_exists:", csv_output_path.exists())
print("saved_csv_size_bytes:", csv_output_path.stat().st_size)

saved_csv_path: /Users/im-youngchan/Desktop/US Financial/data/apple_financials.csv
saved_csv_exists: True
saved_csv_size_bytes: 1492


In [39]:
reloaded_financials_df = pd.read_csv(
    csv_output_path,
    dtype={
        "ticker": "string",
        "cik": "string",
    },
    parse_dates=[
        "period_start",
        "period_end",
        "revenue_filed",
        "net_income_filed",
    ],
)

print(
    "reloaded_shape:",
    reloaded_financials_df.shape,
)
print(
    "reloaded_cik_values:",
    reloaded_financials_df["cik"].unique().tolist(),
)
print(
    "reloaded_fiscal_years:",
    reloaded_financials_df["fiscal_year"].tolist(),
)

display(reloaded_financials_df)

reloaded_shape: (5, 19)
reloaded_cik_values: ['0000320193']
reloaded_fiscal_years: [2021, 2022, 2023, 2024, 2025]


,company_name,ticker,cik,fiscal_year,period_start,period_end,revenue_usd,net_income_usd,form,unit,revenue_concept,net_income_concept,revenue_duration_days,net_income_duration_days,revenue_filed,net_income_filed,revenue_accn,net_income_accn,source
0,Apple Inc.,AAPL,0000320193,2021,2020-09-27,2021-09-25,365817000000,94680000000,10-K,USD,RevenueFromContractWithCustomerExcludingAssess...,NetIncomeLoss,364,364,2023-11-03,2023-11-03,0000320193-23-000106,0000320193-23-000106,SEC EDGAR companyfacts
1,Apple Inc.,AAPL,0000320193,2022,2021-09-26,2022-09-24,394328000000,99803000000,10-K,USD,RevenueFromContractWithCustomerExcludingAssess...,NetIncomeLoss,364,364,2024-11-01,2024-11-01,0000320193-24-000123,0000320193-24-000123,SEC EDGAR companyfacts
2,Apple Inc.,AAPL,0000320193,2023,2022-09-25,2023-09-30,383285000000,96995000000,10-K,USD,RevenueFromContractWithCustomerExcludingAssess...,NetIncomeLoss,371,371,2025-10-31,2025-10-31,0000320193-25-000079,0000320193-25-000079,SEC EDGAR companyfacts
3,Apple Inc.,AAPL,0000320193,2024,2023-10-01,2024-09-28,391035000000,93736000000,10-K,USD,RevenueFromContractWithCustomerExcludingAssess...,NetIncomeLoss,364,364,2025-10-31,2025-10-31,0000320193-25-000079,0000320193-25-000079,SEC EDGAR companyfacts
4,Apple Inc.,AAPL,0000320193,2025,2024-09-29,2025-09-27,416161000000,112010000000,10-K,USD,RevenueFromContractWithCustomerExcludingAssess...,NetIncomeLoss,364,364,2025-10-31,2025-10-31,0000320193-25-000079,0000320193-25-000079,SEC EDGAR companyfacts


In [40]:
if list(reloaded_financials_df.columns) != final_columns:
    raise ValueError("CSV column 순서가 예상과 다릅니다.")

if len(reloaded_financials_df) != 5:
    raise ValueError("CSV row 수가 5가 아닙니다")

if not reloaded_financials_df["cik"].eq(apple_cik).all():
    raise ValueError("CSV의 Apple CIK가 일치하지 않습니다.")

if not reloaded_financials_df["ticker"].eq(APPLE_TICKER).all():
    raise ValueError("CSV의 ticker가 APPL과 일치하지 않습니다.")

if not reloaded_financials_df["form"].eq("10-K").all():
    raise ValueError("CSV에 10-K 이외의 form이 있습니다.")

if not reloaded_financials_df["unit"].eq("USD").all():
    raise ValueError("CSV의 unit이 USD로 통일되지 않았습니다.")

print("csv_validation_passed:", True)

csv_validation_passed: True


In [41]:
first_row = apple_financials_df.iloc[0]
last_row = apple_financials_df.iloc[-1]

revenue_change_usd_b = (
    last_row["revenue_usd"]
    - first_row["revenue_usd"]
) / 1_000_000_000
net_income_change_usd_b = (
    last_row["net_income_usd"]
    - first_row["net_income_usd"]
) / 1_000_000_000

summary_setence = (
    f"Apple의 매출은 fiscal year {int(first_row['fiscal_year'])}의 "
    f"{first_row['revenue_usd'] / 1_000_000-000:.3f} billion USD에서 "
    f"fiscal year {int(last_row['fiscal_year'])}의 "
    f"{last_row['revenue_usd'] / 1_000_000-000:.3f} billion USD로 "
    f"{revenue_change_usd_b:.3f} billion USD 증가했다. "
    f"같은 기간 순이익은 "
    f"{first_row['net_income_usd'] / 1_000_000-000:.3f} billion USD에서 "
    f"{last_row['net_income_usd'] / 1_000_000-000:.3f} billion USD로 "
    f"{net_income_change_usd_b:.3f} billion USD 증가했다. "
    "이 표는 SEC companyfacts의 10-K annual facts를 정리한 것이며, "
    "변화의 원인은 10-K 원문과 주석을 추가로 확인해야 한다."
)

print(summary_setence)

Apple의 매출은 fiscal year 2021의 365817.000 billion USD에서 fiscal year 2025의 416161.000 billion USD로 50.344 billion USD 증가했다. 같은 기간 순이익은 94680.000 billion USD에서 112010.000 billion USD로 17.330 billion USD 증가했다. 이 표는 SEC companyfacts의 10-K annual facts를 정리한 것이며, 변화의 원인은 10-K 원문과 주석을 추가로 확인해야 한다.
